In [ ]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://{GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
!pip install faiss-cpu
!pip install -e .

In [ ]:
!git pull

In [ ]:
%cd data/raw
!cat dataset_part_* > dataset.7z
!7z x dataset.7z
%cd ../..
!ls data/raw/Train

In [ ]:
!pip install -U huggingface_hub
!hf auth login --token {HF_TOKEN}

In [ ]:
%%writefile configs/data/default.yaml
raw_path: data/raw
processed_path: data/processed
eval_size: 0.2
random_state: 36
segmentation: null
overwrite: true

In [ ]:
!python scripts/process_data.py --config configs/data/default.yaml

In [ ]:
!python scripts/update_batch_size.py 32 

In [ ]:
!python scripts/run_pipeline.py \
    configs/model-selection/*.yaml --results \
    results/model-selection/scores.json

In [ ]:
!cat results/model-selection/scores.json | python3 -m json.tool

In [ ]:
import json
import pandas as pd

with open("results/model-selection/scores.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient="index")
df.index.name = "experiment"

df.to_csv("results/model-selection/scores.csv")